# getCCN
Very simple: read through the list of cast sheets from BATS and pull the cruise/cast/niskin information for the BIOS-SCOPE cruises\
Do this as a separate script as I will only need to process the BATS cast sheets once in a while\
Krista Longnecker, 18 September 2026

In [77]:
import os
import re
import pandas as pd
import xlwings as xw #need to use this to make the new sheet to keep the Dalhousie validations
import pdb #use with set_trace() 

In [78]:
#where are the cast sheets?
data_dir = 'seq_data\\Sample_sheets_20260915'
folders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]

In [79]:
alldf = []

for fs in folders:
    files = os.listdir(os.path.join(data_dir,fs))
    
    for fi in files:
        df = pd.read_excel(os.path.join(data_dir,fs,fi),nrows=0)
        fc = df.columns.tolist()
        fc = fc[0].lower()
        if 'bios-scope' in fc:
            #now read in the whole file
            df = pd.read_excel(os.path.join(data_dir,fs,fi),skiprows = 1)
            #delete the row with the BATS sample numbers:
            df = df.drop(df.index[0]).reset_index(drop = True)

            #look for BS_eDNA ...and pull sample numbers
            df = df[pd.notna(df['BS_eDNA'])].reset_index(drop = True)

            #make the NewID, will have to pad Niskin
            df['NewID'] = df['% cast'].astype(str) + df['Niskin'].astype(str).str.zfill(2)

            #I think the easy export is to write a new Excel file with what I need...may change my mind later
            df = df[['NewID','BS_eDNA','Depth','Flag']]
            #drop the last row that repeats the top row
            df = df.drop(df.index[-1]) 
            #add back in cruise as that will make the matching easier
            df['cruise'] = fi[:5]
            #put the sheets one folder up as I don't want to go digging in individual batches of cast sheets
            df.to_excel(os.path.join('seq_data',fi[:5] + '_biosscope.xlsx'))
            
            #also merge all of these into a single file...that will be easier to use later
            alldf.append(df)
        
dfc = pd.concat(alldf,ignore_index=True)
dfc.to_excel(os.path.join('seq_data','biosscope_combinedCCN.xlsx'),index=False)